# Day 4: Giving Your Chatbot a Memory

**LangChain Hands-on Series | Day 4 of 10**
Trainer: Ajeetkumar

---

On Day 1 we discovered something important: **the model forgets everything between calls**. We worked around it by keeping a Python list and sending it every time.

That is fine for a demo, but a real chatbot has problems the list does not solve:

- **Many users at once.** Asha's chat must never get mixed with Rahul's.
- **Long chats.** After 200 messages the history no longer fits in the model's context window, and every call gets slower and more expensive.
- **Restarts.** If your server restarts, the list in memory is gone.

Today we solve all three.

```
              user message
                   |
                   v
   +---------------------------------+        +----------------------+
   |  load history for this session  | <----> |  history store       |
   +---------------------------------+        |  (memory / file / DB)|
                   |                          +----------------------+
                   v                                     ^
     prompt = system + history + new message             |
                   |                                     |
                   v                                     |
                  LLM  -->  answer  ---- save both ------+
```

**Mini project at the end:** a study-buddy chatbot with separate sessions, a size limit, and a history that survives a restart.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

load_dotenv()
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
print("Ready. Model:", MODEL_NAME)

## 1. Quick reminder: the model really does forget

In [ ]:
print(llm.invoke("Hi! My name is Priya and I live in Pune.").content)
print("---")
print(llm.invoke("Where do I live?").content)       # a completely separate call

Two separate calls, two strangers. Now let's fix it properly.

## 2. A memory-ready prompt

Remember `MessagesPlaceholder` from Day 2? This is exactly what it is for. The prompt has a slot called `history` where all earlier messages will go.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly study buddy. Keep answers short and encouraging."),
    MessagesPlaceholder("history"),          # earlier messages are inserted here
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

## 3. A proper history object

Instead of a raw Python list, LangChain has **chat message history** classes. They all share the same small interface, so you can swap an in-memory one for a database-backed one later without changing your chain.

| Method / attribute | Purpose |
|--------------------|---------|
| `.messages` | all stored messages |
| `.add_user_message(text)` | store what the user said |
| `.add_ai_message(text)` | store what the bot said |
| `.clear()` | forget everything |

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory

history = InMemoryChatMessageHistory()

history.add_user_message("Hi! My name is Priya and I live in Pune.")
history.add_ai_message("Nice to meet you, Priya! Pune is a lovely city.")

history.messages

In [ ]:
answer = chain.invoke({"history": history.messages, "question": "Where do I live?"})
print(answer)

It works, but notice we still had to save each message ourselves. Forgetting a single `add_ai_message` call would silently break the conversation. Let's hand that job over to LangChain.

## 4. RunnableWithMessageHistory - automatic memory

`RunnableWithMessageHistory` wraps any chain and does the boring part for you on every call:

1. looks up the history for this `session_id`
2. puts it into the `history` slot of the prompt
3. runs the chain
4. saves both the new question and the answer back to that history

All you provide is a function that says "here is the history object for session X".

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}      # session_id -> history object. In production this would be Redis, Postgres, etc.

def get_session_history(session_id):
    if session_id not in store:                      # first message from this session?
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [ ]:
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",       # which input key holds the new user message
    history_messages_key="history",      # which prompt slot receives the history
)

Now each call needs a `session_id` in the config. Think of it as the chat window ID.

In [ ]:
priya = {"configurable": {"session_id": "priya"}}

print(chatbot.invoke({"question": "Hi! I'm Priya, and I'm learning LangChain."}, config=priya))

In [ ]:
print(chatbot.invoke({"question": "What am I learning, and what's my name?"}, config=priya))

**Pause and predict:** a second user, Rahul, now opens his own chat and asks "What is my name?". What will the bot say?

In [ ]:
rahul = {"configurable": {"session_id": "rahul"}}

print(chatbot.invoke({"question": "What is my name?"}, config=rahul))

<details>
<summary>Click to see the explanation</summary>

The bot does not know. Rahul's session has its own empty history, completely separate from Priya's. This separation is exactly what you need when many users talk to the same bot.
</details>

In [ ]:
for session_id, hist in store.items():
    print(f"Session '{session_id}': {len(hist.messages)} messages")
    for m in hist.messages:
        print(f"    {m.type:>5}: {m.content[:70]}")

## 5. The growing-history problem

Every message we keep gets sent again on **every** call. So the cost of each call keeps climbing as the chat goes on. Let's simulate a 15-turn chat and measure how big the history gets.

We use `count_tokens_approximately`, a quick estimate that does not need an API call.

In [ ]:
from langchain_core.messages.utils import count_tokens_approximately

demo_store = {}
demo_bot = RunnableWithMessageHistory(
    chain,
    lambda sid: demo_store.setdefault(sid, InMemoryChatMessageHistory()),
    input_messages_key="question",
    history_messages_key="history",
)

tokens_per_turn = []
for turn in range(1, 16):
    demo_bot.invoke({"question": f"Give me study tip number {turn} for learning Python."},
                    config={"configurable": {"session_id": "long_chat"}})
    msgs = demo_store["long_chat"].messages
    tokens_per_turn.append(count_tokens_approximately(msgs))     # size of history after this turn

print("History size after the last turn:", tokens_per_turn[-1], "tokens (approx.)")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.plot(range(1, 16), tokens_per_turn, marker="o", color="#C44E52")
plt.xlabel("Turn number")
plt.ylabel("Tokens sent as history")
plt.title("Without limits, every call gets bigger")
plt.grid(alpha=0.3)
plt.show()

That line only goes up. In a long chat you will eventually hit the model's context limit, and long before that you are paying for the same old messages again and again.

There are two common fixes:

| Strategy | Idea | Good for |
|----------|------|----------|
| **Trimming** | keep only the most recent messages | most chatbots; cheap and simple |
| **Summarising** | replace old messages with a short summary | long chats where older details still matter |

## 6. Trimming with `trim_messages`

`trim_messages` keeps the most recent messages that fit within a token budget. The settings read almost like English:

In [ ]:
from langchain_core.messages import trim_messages

trimmer = trim_messages(
    max_tokens=120,                          # budget for the history
    strategy="last",                         # keep the newest messages, drop the oldest
    token_counter=count_tokens_approximately,
    include_system=True,                     # never drop the system message
    start_on="human",                        # the kept history should start with a user message
)

long_history = demo_store["long_chat"].messages
trimmed = trimmer.invoke(long_history)

print("Before trimming:", len(long_history), "messages")
print("After trimming :", len(trimmed), "messages")

Why `start_on="human"`? If trimming cut in the middle of a pair, the history could start with an AI reply to a question the model can no longer see. That confuses it. Starting on a human message keeps the conversation sensible.

Now plug the trimmer **into the chain**, so the history is trimmed right before every model call. We trim inside the chain, not in the store, so the full history is still saved - we simply send less of it.

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

trimmed_chain = (
    RunnablePassthrough.assign(history=itemgetter("history") | trimmer)   # trim, then continue as before
    | prompt
    | llm
    | StrOutputParser()
)

trim_store = {}
trimmed_bot = RunnableWithMessageHistory(
    trimmed_chain,
    lambda sid: trim_store.setdefault(sid, InMemoryChatMessageHistory()),
    input_messages_key="question",
    history_messages_key="history",
)

In [ ]:
sent_tokens = []
for turn in range(1, 16):
    cfg = {"configurable": {"session_id": "trimmed_chat"}}
    trimmed_bot.invoke({"question": f"Give me study tip number {turn} for learning Python."}, config=cfg)
    sent = trimmer.invoke(trim_store["trimmed_chat"].messages)       # what would be sent next time
    sent_tokens.append(count_tokens_approximately(sent))

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(range(1, 16), tokens_per_turn, marker="o", label="No trimming", color="#C44E52")
plt.plot(range(1, 16), sent_tokens, marker="s", label="With trim_messages", color="#55A868")
plt.xlabel("Turn number")
plt.ylabel("Tokens sent as history")
plt.title("Trimming puts a ceiling on cost")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The green line flattens out. The trade-off: the bot forgets the oldest part of the conversation. Let's check that honestly.

In [ ]:
trim_store.clear()
cfg = {"configurable": {"session_id": "forget_test"}}

trimmed_bot.invoke({"question": "My favourite colour is teal. Please remember it."}, config=cfg)
for i in range(8):
    trimmed_bot.invoke({"question": f"Tell me a one-line fact about planet number {i + 1}."}, config=cfg)

print(trimmed_bot.invoke({"question": "What is my favourite colour?"}, config=cfg))

It probably forgot. That is what summarising is for.

## 7. Summarising old messages

Instead of throwing old messages away, we ask the LLM to squeeze them into a short summary, and keep only that summary plus the latest few messages.

```
 [m1, m2, m3, m4, m5, m6, m7, m8]
          |
          v
 [ SystemMessage("Summary so far: user likes teal, asked about planets...") , m7, m8 ]
```

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    MessagesPlaceholder("messages"),
    ("human", "Summarise the conversation above in 3 short bullet points. "
              "Keep personal details like names and preferences."),
])
summariser = summary_prompt | llm | StrOutputParser()

def summarise_history(history, keep_last=2):
    """Replace all but the last `keep_last` messages with a single summary message."""
    msgs = history.messages
    if len(msgs) <= keep_last:
        return                                            # nothing to do yet
    old, recent = msgs[:-keep_last], msgs[-keep_last:]
    summary = summariser.invoke({"messages": old})
    history.clear()
    history.add_message(SystemMessage(content=f"Summary of the earlier conversation:\n{summary}"))
    history.add_messages(recent)

In [ ]:
hist = trim_store["forget_test"]
print("Messages before summarising:", len(hist.messages))

summarise_history(hist)

print("Messages after summarising :", len(hist.messages))
print("\n" + hist.messages[0].content)

Long histories shrink to a summary plus a couple of recent messages, and important details like the favourite colour survive (as long as the summary kept them). On Day 8 you will see that LangChain agents can do this automatically using `SummarizationMiddleware`.

## 8. Memory that survives a restart

Everything so far lives in Python memory. Restart the kernel and it is gone. A real app stores history in a database (Redis, Postgres, MongoDB...). LangChain has ready-made classes for many of them.

To understand how they work inside, let's build a tiny one ourselves that saves each session to a JSON file. It only needs three things: read messages, add messages, clear.

In [ ]:
import json
from pathlib import Path
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import messages_from_dict, messages_to_dict

class JSONFileHistory(BaseChatMessageHistory):
    """Stores one session's messages in a JSON file."""

    def __init__(self, session_id, folder="chat_histories"):
        Path(folder).mkdir(exist_ok=True)
        self.path = Path(folder) / f"{session_id}.json"

    @property
    def messages(self):
        if not self.path.exists():
            return []
        return messages_from_dict(json.loads(self.path.read_text()))     # JSON -> message objects

    def add_messages(self, messages):
        all_messages = self.messages + list(messages)
        self.path.write_text(json.dumps(messages_to_dict(all_messages), indent=2))   # message objects -> JSON

    def clear(self):
        self.path.unlink(missing_ok=True)

In [ ]:
file_bot = RunnableWithMessageHistory(
    chain,
    lambda session_id: JSONFileHistory(session_id),     # the only line that changed
    input_messages_key="question",
    history_messages_key="history",
)

cfg = {"configurable": {"session_id": "student_42"}}
JSONFileHistory("student_42").clear()                    # start fresh for this demo

print(file_bot.invoke({"question": "Hi, I'm Kiran. I'm preparing for an AWS exam."}, config=cfg))

Now imagine the server restarted. We create a **brand-new** bot object - nothing in Python memory - and continue the same session.

In [ ]:
restarted_bot = RunnableWithMessageHistory(
    chain,
    lambda session_id: JSONFileHistory(session_id),
    input_messages_key="question",
    history_messages_key="history",
)

print(restarted_bot.invoke({"question": "Which exam am I preparing for?"}, config=cfg))

In [ ]:
print(Path("chat_histories/student_42.json").read_text()[:600])    # peek at the saved file

## Mini project: study buddy with all three fixes

Let's combine everything: separate sessions, trimming, and file storage. We also wrap it in a small `chat()` helper so it feels like using a real chatbot.

In [ ]:
study_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are StudyBuddy, a motivating tutor for students learning AI. "
               "Use the student's name when you know it. Keep replies under 60 words."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])

study_chain = (
    RunnablePassthrough.assign(history=itemgetter("history") | trimmer)
    | study_prompt
    | llm
    | StrOutputParser()
)

study_buddy = RunnableWithMessageHistory(
    study_chain,
    lambda sid: JSONFileHistory(sid),
    input_messages_key="question",
    history_messages_key="history",
)

def chat(session_id, message):
    reply = study_buddy.invoke({"question": message}, config={"configurable": {"session_id": session_id}})
    print(f"You      : {message}")
    print(f"StudyBuddy: {reply}\n")

In [ ]:
JSONFileHistory("anita").clear()

chat("anita", "Hello! I'm Anita. I keep getting confused between RAG and fine-tuning.")
chat("anita", "Can you give me a simple analogy?")
chat("anita", "Thanks! Can you remind me what I said I was confused about?")

### Your turn

Start your own session below. Tell StudyBuddy your name and a topic you find hard, ask two follow-up questions, then restart the kernel, re-run the setup and mini-project cells, and see if it still remembers you.

In [ ]:
chat("my_session", "Hi! My name is ... and I find ... difficult.")

## Common mistakes on Day 4

| What you see | What usually went wrong |
|--------------|-------------------------|
| `ValueError: Missing keys ['session_id']` | You forgot `config={"configurable": {"session_id": ...}}` |
| The bot remembers nothing | `history_messages_key` does not match the `MessagesPlaceholder` name |
| Two users see each other's chat | Both are using the same `session_id` |
| Context length exceeded error | Add `trim_messages` or summarisation |

## Quick recap

- Models are stateless; memory means storing messages and sending them back.
- `InMemoryChatMessageHistory` stores messages; `RunnableWithMessageHistory` loads and saves them automatically per `session_id`.
- Histories grow without limit. Trim them (`trim_messages`) or summarise them.
- Swap the storage class to persist history; the chain itself does not change.

> Looking ahead: for **agents** (Day 8), LangChain uses a newer approach called a *checkpointer* with a `thread_id`. The idea is identical - a separate saved conversation per ID - so everything you learned today carries over directly.

## Practice before Day 5

1. Change the trimmer to keep only the last 4 messages using `token_counter=len` and `max_tokens=4`. (With `len`, each message counts as one "token".)
2. Call `summarise_history` automatically whenever a session has more than 10 messages.
3. Add a `/reset` command to `chat()` that clears the session.
4. Plot the number of messages stored per session in `store`.

**Tomorrow:** we teach the model about *your* documents - loading files, splitting them, and searching them by meaning instead of keywords.